In [5]:
import os
import re
import requests
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

In [ ]:
def ingest(book_title):
    print(f"Searching for book : {book_title}")
    url = f"https://gutendex.com/books/?search={book_title.replace(' ', '%20')}"
    os.environ['PINECONE_API_KEY'] = 'YOUR_PINECONE_API_KEY'
    response = requests.get(url).json()
    if response['count'] == 0:
        return "Book not found in library"
    book_info = response['results'][0]
    title = book_info['title']
    format = book_info['formats']
    text_url = None
    for key, fmt_url in format.items():
        if 'text/plain' in key:
            text_url = fmt_url
    if not text_url:
        return f"Plain text for {title} not found"
    print(f"Found {title}, downloading the pdf from {text_url}")
    text_response = requests.get(text_url)
    title = re.sub(r'[\\/*?:"<>|]', "", title)
    file_path = f"temp_{title.replace(' ','_')}.txt"
    with open(file_path,'w',encoding = 'utf-8') as file:
        file.write(text_response.text)
    print("Loading and chunking text")
    loader = TextLoader(file_path,encoding ='utf-8')
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
    chunks = text_splitter.split_documents(documents)
    for chunk in chunks:
        chunk.metadata = {'book_title':title}
    print("Embedding and saving to Pinecone")
    embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
    index_name = "enchanted-library"
    vector_db = PineconeVectorStore.from_documents(
        documents=chunks,
        embedding=embedding_model,
        index_name=index_name
    )
    os.remove(file_path)
    print("Successfully added",title,'to AI library')

In [ ]:
ingest("Bhagavad Gita")
ingest("Pride and Prejudice")
ingest("Frankenstein")
ingest("Little Women")
ingest("Sense and Sensibility")
ingest("The Mahabharata")
ingest("Crime and Punishment")
ingest("The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4)")

Searching for book : Bhagavad Gita
Found Bhagavadgita — Des Erhabenen Sang: Religiöse Stimmen der Völker: Die Religion des alten Indien II, downloading the pdf from https://www.gutenberg.org/files/33186/33186-8.txt
Loading and chunking text
Embedding and saving to Pinecone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5566.11it/s]


Successfully added Bhagavadgita — Des Erhabenen Sang Religiöse Stimmen der Völker Die Religion des alten Indien II to AI library
Searching for book : Pride and Prejudice
Found Pride and Prejudice, downloading the pdf from https://www.gutenberg.org/ebooks/1342.txt.utf-8
Loading and chunking text
Embedding and saving to Pinecone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4353.44it/s]


Successfully added Pride and Prejudice to AI library
Searching for book : Frankenstein
Found Frankenstein; or, the modern prometheus, downloading the pdf from https://www.gutenberg.org/ebooks/84.txt.utf-8
Loading and chunking text
Embedding and saving to Pinecone


...

Successfully added The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4) to AI library
Searching for book : The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 3 (of 4)
Found The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4), downloading the pdf from https://www.gutenberg.org/files/71326/71326-0.txt
Loading and chunking text
Embedding and saving to Pinecone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5529.49it/s]


Successfully added The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4) to AI library
Searching for book : The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 4 (of 4)
Found The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4), downloading the pdf from https://www.gutenberg.org/files/71326/71326-0.txt
Loading and chunking text
Embedding and saving to Pinecone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5726.66it/s]


Successfully added The Yoga-Vasishtha Maharamayana of Valmiki, Vol. 1 (of 4) to AI library


In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

os.environ['PINECONE_API_KEY'] = 'YOUR_PINECONE_API_KEY'

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_db = PineconeVectorStore(index_name="enchanted-library", embedding=embedding_model)

docs = vector_db.similarity_search("Who is the main character?", k=1)
print(f"Database contains content! Sample snippet: {docs[0].page_content[:100]}...")

c:\Users\KHUSHI CHADHA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6157.19it/s]


Database contains content! Sample snippet: O hero, wait a little.’



“Vaisampayana continued, ‘Thus addressed (by Arjuna), Karna the adopted

...
